# Queimadas no Pará: exploração e resultados

Este notebook **não processa dados**: só lê as tabelas que o pipeline gerou em `data/silver/` e `data/gold/` e mostra o que elas dizem.
Antes de abrir, rode o pipeline na raiz do repositório (ver README):

```bash
python -m src.ingestion.run_bronze
python -m src.silver.run_silver
python -m src.gold.run_gold
python -m src.ml.modelo
```

Os gráficos também são salvos em `docs/figuras/` para usar nos slides.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

from src.gold import util as gold
from src.silver import util as silver

FIGURAS = RAIZ / "docs" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

AZUL, LARANJA, VERDE_AGUA = "#2a78d6", "#eb6834", "#1baf7a"
CINZA, TEXTO, TEXTO_2, GRADE, FUNDO = "#c3c2b7", "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FUNDO, "axes.facecolor": FUNDO, "savefig.facecolor": FUNDO,
    "axes.edgecolor": GRADE, "axes.labelcolor": TEXTO_2, "xtick.color": TEXTO_2, "ytick.color": TEXTO_2,
    "axes.titlecolor": TEXTO, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "figure.dpi": 110,
})


def milhar(v, _=None):
    return f"{v:,.0f}".replace(",", ".")


focos = silver.ler("foco_calor")[["cod_ibge", "lat", "lon", "ano", "ano_mes"]]
dim = silver.ler("dim_municipio")
mensal = gold.ler("focos_municipio_mes")
base = gold.ler("ml_base")
meta = gold.ler_meta("ml_base")
metricas = json.loads((gold.GOLD_DIR / "modelo_metricas.json").read_text(encoding="utf-8"))
ranking = gold.ler("ranking_prioridade")

print(f"{milhar(len(focos))} focos no Pará | {milhar(len(mensal))} linhas município-mês | {milhar(len(base))} linhas na ml_base")

## 1. Quanto o Pará queima por ano

Contagem do satélite de referência (AQUA_M-T), que mantém a série comparável entre os anos. 2026 fica de fora porque o ano ainda não acabou.

2024 é o ano extremo da série. Isso ajuda a explicar por que o modelo foi um pouco pior no teste (2024 a 2026) do que na validação (2022 e 2023).

In [ ]:
por_ano = mensal[mensal["ano"] <= 2025].groupby("ano")["n_focos"].sum()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(por_ano.index, por_ano.values, color=AZUL, width=0.7)
pico = por_ano.idxmax()
ax.annotate(f"{pico}: {milhar(por_ano[pico])} focos", (pico, por_ano[pico]),
            xytext=(0, 6), textcoords="offset points", ha="center", color=TEXTO)
ax.set_title("Focos de calor no Pará por ano")
ax.set_ylabel("focos (satélite de referência)")
ax.set_xticks(por_ano.index)
ax.tick_params(axis="x", labelrotation=45)
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(milhar)
fig.tight_layout()
fig.savefig(FIGURAS / "focos_por_ano.png", dpi=150)
plt.show()

por_ano.rename("focos").to_frame().T

## 2. Sazonalidade: o fogo tem época

Média de focos em cada mês do calendário, de 2013 a 2025. Quase todo o fogo acontece na estiagem, de julho a novembro.

Por isso o modelo recebe o mês-alvo como feature (seno e cosseno) e a média histórica do mês-alvo, e por isso os resultados também são reportados só para a estiagem.

In [ ]:
NOMES_MES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
ESTIAGEM = {7, 8, 9, 10, 11}

por_mes = (mensal[mensal["ano"] <= 2025].groupby(["ano", "mes_num"])["n_focos"].sum()
           .groupby("mes_num").mean())
cores = [AZUL if m in ESTIAGEM else CINZA for m in por_mes.index]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(NOMES_MES, por_mes.values, color=cores, width=0.7)
ax.set_title("Média de focos por mês do ano (2013 a 2025)")
ax.set_ylabel("focos por mês, média")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(milhar)
ax.text(0.02, 0.9, "azul: estiagem (jul a nov)", transform=ax.transAxes, color=TEXTO_2)
fig.tight_layout()
fig.savefig(FIGURAS / "sazonalidade.png", dpi=150)
plt.show()

participacao = por_mes[list(ESTIAGEM)].sum() / por_mes.sum()
print(f"A estiagem concentra {participacao:.0%} dos focos de um ano típico.")

## 3. Onde o fogo se concentra

Densidade de todos os focos de 2013 a set/2026 (quanto mais escuro, mais focos). Os círculos marcam os 15 municípios priorizados para outubro de 2026, na posição média dos focos de cada um.

In [ ]:
sequencial = LinearSegmentedColormap.from_list("azul", ["#dbe9fa", AZUL, "#0d3a73"])

centro = focos.groupby("cod_ibge")[["lat", "lon"]].median()
top = ranking[ranking["top_15"]].merge(centro, left_on="cod_ibge", right_index=True)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
hb = ax.hexbin(focos["lon"], focos["lat"], gridsize=70, bins="log", cmap=sequencial, mincnt=1, linewidths=0)
ax.scatter(top["lon"], top["lat"], s=90, facecolors="none", edgecolors=LARANJA, linewidths=2, zorder=3)
for _, linha in top.head(5).iterrows():
    ax.annotate(f"{linha['posicao']}. {linha['nome']}", (linha["lon"], linha["lat"]),
                xytext=(7, 4), textcoords="offset points", color=TEXTO, fontsize=9)
ax.set_title("Focos de calor no Pará (2013 a 2026) e top 15 de outubro")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_aspect("equal")
ax.grid(False)
barra = fig.colorbar(hb, ax=ax, shrink=0.7)
barra.set_label("focos por célula (escala log)", color=TEXTO_2)
fig.tight_layout()
fig.savefig(FIGURAS / "mapa_focos.png", dpi=150)
plt.show()

## 4. O label e o desbalanceamento

`critico = 1` quando o município tem 39 focos ou mais no mês seguinte ao t0 (percentil 90 do treino). Só cerca de 10% das linhas são positivas, e quase todas caem na estiagem. Por isso a acurácia não serve como métrica: dizer sempre "não crítico" acertaria 90%.

In [ ]:
print(meta["label"])
display(pd.DataFrame(meta["particoes"]).set_index("particao"))

treino = base[base["particao"] == "treino"].copy()
treino["mes_alvo_num"] = pd.PeriodIndex(treino["mes_alvo"], freq="M").month
taxa = treino.groupby("mes_alvo_num")["critico"].mean()

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(NOMES_MES, taxa.values, color=[AZUL if m in ESTIAGEM else CINZA for m in taxa.index], width=0.7)
ax.set_title("Fração de municípios críticos por mês-alvo (treino)")
ax.set_ylabel("críticos")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIGURAS / "label_por_mes.png", dpi=150)
plt.show()

## 5. O modelo contra os baselines (teste: meses-alvo de 2024 a 2026)

Todo mês, os 15 municípios de maior score recebem equipe. As métricas:

- **recall**: das situações críticas reais, quantas estavam no top 15;
- **teto**: o máximo que um modelo perfeito alcançaria, porque em setembro há mais de 15 municípios críticos;
- **precisão**: das 15 equipes, quantas foram para um município que ficou crítico;
- **focos cobertos**: fração de todos os focos do mês que caíram nos 15 municípios.

O baseline principal é "priorize quem mais queimou no mesmo mês do ano anterior".

In [ ]:
NOMES = {
    "regressao_logistica": "Regressão logística (modelo)",
    "baseline_mesmo_mes_ano_anterior": "Baseline: mesmo mês do ano anterior",
    "baseline_ultimo_mes": "Baseline: último mês",
}
linhas = []
for chave, nome in NOMES.items():
    r = metricas["teste"][chave]
    linhas.append({
        "score": nome,
        "PR-AUC": r["pr_auc"],
        "recall top-15": r["top_k"]["recall"],
        "teto": r["top_k"]["recall_maximo_possivel"],
        "precisão top-15": r["top_k"]["precisao"],
        "focos cobertos": r["top_k"]["cobertura_focos"],
        "recall estiagem": r["top_k_temporada"]["recall"],
        "precisão estiagem": r["top_k_temporada"]["precisao"],
        "falsos positivos/mês estiagem": r["top_k_temporada"]["falsos_positivos_por_mes"],
    })
pd.DataFrame(linhas).set_index("score").round(3)

In [ ]:
CORES = {"regressao_logistica": AZUL, "baseline_mesmo_mes_ano_anterior": LARANJA, "baseline_ultimo_mes": VERDE_AGUA}

fig, ax = plt.subplots(figsize=(8, 4.2))
for chave, cor in CORES.items():
    curva = metricas["teste"][chave]["curva_recall"]
    ks, valores = [int(k) for k in curva], list(curva.values())
    ax.plot(ks, valores, color=cor, linewidth=2, marker="o", markersize=7, label=NOMES[chave])
    ax.annotate(f"{valores[-1]:.0%}", (ks[-1], valores[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", color=TEXTO)
ax.axvline(metricas["k"], color=TEXTO_2, linestyle="--", linewidth=1)
ax.text(metricas["k"] + 0.4, 0.05, "15 equipes", color=TEXTO_2)
ax.set_title("Situações críticas cobertas por número de equipes (teste)")
ax.set_xlabel("municípios priorizados por mês (k)")
ax.set_ylabel("recall")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURAS / "recall_por_k.png", dpi=150)
plt.show()

## 6. O que o modelo usa

Importância por permutação no teste: quanto a PR-AUC cai quando uma feature é embaralhada. O risco do mês seguinte é explicado quase todo pelo histórico do município naquele mês e pelo quanto ele já está queimando agora. As variáveis do IBGE quase não ajudam.

In [ ]:
importancia = pd.Series(metricas["importancia_permutacao_teste"]).sort_values().tail(8)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(importancia.index, importancia.values, color=AZUL, height=0.6)
ax.set_title("Importância das features (queda de PR-AUC no teste)")
ax.set_xlabel("queda de PR-AUC ao embaralhar")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIGURAS / "importancia_features.png", dpi=150)
plt.show()

## 7. A decisão: ranking para outubro de 2026

t0 = 30/09/2026. Os 15 primeiros são a recomendação de onde posicionar as brigadas.

In [ ]:
colunas = {"posicao": "posição", "nome": "município", "regiao_intermediaria": "região",
           "score": "score", "focos_m0": "focos em set/2026", "focos_mes_alvo_ano_anterior": "focos em out/2025"}
display(ranking[ranking["top_15"]][list(colunas)].rename(columns=colunas).set_index("posição").round(3))

conf = metricas["conferencia_mes_em_curso"]
print(f"Conferência (não é avaliação, o mês ainda não acabou): até {conf['dias_observados_ate']}, "
      f"os 15 do ranking concentram {conf['focos_parciais_no_top_k']} de {conf['focos_parciais_total']} focos de outubro "
      f"({conf['focos_parciais_no_top_k'] / conf['focos_parciais_total']:.0%}) e "
      f"{conf['top_k_em_comum_com_o_parcial']} deles estão entre os 15 que mais queimaram até agora.")